# Discussion Group Week 4 - Exercise
In this notebook, you will be implementing a neural network using Pytorch. The goal is to create a simple feedforward neural network that can classify a synthetic dataset.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn
import matplotlib.pyplot as plt

## Dataset
Below is the code to generate a synthetic dataset for classification. The dataset consists of five classes that are not linearly separable.

In [ ]:
from sklearn.datasets import make_blobs

# Generate synthetic dataset with 5 classes
X, y = make_blobs(
    n_samples=1000,
    centers=5,        # number of labels/classes
    n_features=2,     # number of features (2 makes it easy to plot)
    cluster_std=1.7,  # spread of clusters
    random_state=42
)

plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

## Pytorch Dataset

In [ ]:
# TODO Create a custom PyTorch dataset called CustomDataset

## Pytorch Neural Network

In [ ]:
# TODO Create the custom MLP called CustomNetwork

## Dataloader and Training Loop

In [ ]:
batch_size = 32
learning_rate = 1e-3
num_epochs = 35

In [ ]:
# Create dataset
dataset = CustomDataset(X, y)

# Randomly split the dataset into train, validation, test sets
train_set, val_set, test_set = random_split(dataset, [0.8, 0.1, 0.1])

# TODO Create dataloaders from train, val, test sets (call them train_dataloader, val_dataloader, test_dataloader)

In [ ]:
model = CustomNetwork()
criterion = # TODO Choose an appropriate loss function for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
# maintain history of train/val losses
train_losses = []
val_losses = []

for epoch in range(num_epochs):                                # In each epoch the entire training set is used for optimization 
    
    batch_loss = []
    for batch_X, batch_y in train_dataloader:                   # training set is batched
        
        y_pred = model(batch_X)                                 # predict y
        loss = criterion(y_pred, batch_y)                       # compute loss based on ground truth y and predict y 

        batch_loss.append(loss.item())

        optimizer.zero_grad()                                   # reset gradients to zero
        loss.backward()                                         # backpropagation
        optimizer.step()                                        # perform optimization step

    train_losses.append(sum(batch_loss) / len(batch_loss))
    
    with torch.no_grad():                                       # within this block, no gradients are computed

        batch_loss = []
        for batch_X, batch_y in val_dataloader:
            
            y_pred = model(batch_X)
            loss = criterion(y_pred, batch_y)

            batch_loss.append(loss.item())

        val_losses.append(sum(batch_loss) / len(batch_loss))

    print(f"{epoch=} \ttrain_loss = {train_losses[-1]:.4f}\tval_loss = {val_losses[-1]:.4f}")

## Visualize the results

In [ ]:
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.legend()
plt.show()

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min(), X[:, 0].max(), 100), 
    np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
    )

grid = np.vstack([xx.flatten(), yy.flatten()]).T

predictions = model(torch.tensor(grid, dtype=torch.float32)).detach().numpy()
predicted_classes = np.argmax(predictions, axis=1)

plt.contourf(xx, yy, predicted_classes.reshape(xx.shape), alpha=0.3, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k')
plt.show()


## Test metric

In [ ]:
test_loss = 0

with torch.no_grad():

    for batch_X, batch_y in test_dataloader:
        
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)

        test_loss += loss.item()

test_loss / len(test_dataloader)